# Fig 3c and Supp Fig 3b: overall survival of the WES CNV clusters

Survival analysis between CNA clusters, using covariates: Restricted Mean Survival Time
(RMST, τ = 3 years) of each cluster vs the rest, and Kaplan-Meier curves.

- **Fig 3c**: KM of CNV cluster 2 vs the other clusters with the adjusted RMST difference.
- **Supp Fig 3b**: forest of the adjusted RMST difference of each cluster vs the rest.
- Global KM of the four clusters and the global log-rank test (not a panel).

Origin: analysis/survival/wes_profile_os_rmst.ipynb (Spencer Watson).

Model: clinical rows (loader exclusions, the 13 Charité patients of SURVIVAL_EXCLUSION
removed) joined to the 265 clustered samples by block, complete cases, adjusted for
**MGMT, age, sex and p53** in place of the collection site (Spencer Watson's revision of the
model). p53 is missing for many patients, so the model n (173) is below the published 247.

Comparison with the published numbers: the **published model** (age, MGMT, sex, site;
cluster 2 vs rest +3.35 months [0.28, 6.43], p = 0.0326, n = 247 of 253 rows) is run next to
it and reproduces exactly; the last table shows both models next to the published numbers.

In [ ]:
import pandas as pd
import numpy as np
from unittest import mock

# Plotting and Survival packages
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

# Project-specific imports
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter
from gbmcollaboration import SURVIVAL_EXCLUSION
from gbmcollaboration.paper import output
from gbmcollaboration.paper.ids import patient_of

# Rpy2 for RMST via survRM2
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

from gbmcollaboration.definitions import AVG_NUM_DAYS_PER_YEAR
from gbmcollaboration.utils.viz_survival import kmplot
from gbmcollaboration.viz import forest_plot

# ---------------------------------------------------------------------------
# DATA LOADING FUNCTIONS
# ---------------------------------------------------------------------------
def load_clinical_data() -> pd.DataFrame:
    """Loads and concatenates tabular clinical data from the 3 main centers."""
    merged_df = []
    centers =[CHUVDataCenter, UKERDataCenter, CRDataCenter]

    for center in centers:
        center_data_dict = center.load_tabular(
            exclude_idh1_mut=True,
            exclude_patient_ids=True,
            exclude_relapse=True
        )
        df = center_data_dict['clinical']['processed gbm clinical']

        if center == CRDataCenter:
            df = df.drop(index=SURVIVAL_EXCLUSION)

        print(f"\n--- Loading the clinical data from {center.name} ---")
        print(df['Sample collection chronology'].value_counts())
        print(f"Found {len(df)} primary cases")
        merged_df.append(df)

    df_concat = pd.concat(merged_df, ignore_index=False)
    print("\nDataframes from the three centers have been merged.")
    return df_concat


def load_and_merge_cluster_data(clinical_df: pd.DataFrame, endpoint: str,
                                covariate_cols=None) -> tuple[pd.DataFrame, str]:
    """Loads clustering annotations and merges them with the clinical dataframe."""
    covariate_cols = COVARIATE_COLS if covariate_cols is None else covariate_cols
    df_wes = pd.read_csv(CLUSTER_CSV)
    df_wes.set_index("orig.ident", inplace=True)

    cluster_col = "cnv_leiden"

    # Map categorical covariates
    mapping = {
        "MGMT promoter methylation": {"Unmethylated": 0, "Methylated": 1},
        "Administrative gender": {"Male": 0, "Female": 1},
        "Site code": {"CH": 0, "ER": 1, "CR": 2},
        "p53": {"Negative": 0, "Positive": 1},
    }

    df_surv_cov = clinical_df[[endpoint] + covariate_cols].copy()
    df_surv_cov = df_surv_cov.replace(mapping).infer_objects()

    df_cluster = df_wes[[cluster_col]].merge(
        df_surv_cov, how="inner", left_index=True, right_index=True
    ).copy()

    df_cluster[cluster_col] = df_cluster[cluster_col].astype(int)

    # Build time/event in standard lifelines format
    df_cluster[f"{endpoint}.time"] = df_cluster[endpoint].abs()
    df_cluster[f"{endpoint}.event"] = (df_cluster[endpoint] > 0).astype(int)
    df_cluster[f'{endpoint}.time.days'] = df_cluster[f'{endpoint}.time'] * AVG_NUM_DAYS_PER_YEAR

    print(f"\nMerged df shape: {df_cluster.shape}")
    print("Cluster counts:\n", df_cluster[cluster_col].value_counts().sort_index())
    return df_cluster, cluster_col


# ---------------------------------------------------------------------------
# ANALYSIS FUNCTIONS
# ---------------------------------------------------------------------------
def rasterize_ci_bands(ax):
    """Rasterize the KM confidence bands (fill_between) so their transparency survives in the PDF."""
    for coll in ax.collections:
        if isinstance(coll, PolyCollection):
            coll.set_rasterized(True)


def run_rmst_and_km_cluster_vs_rest(df_cluster: pd.DataFrame, cluster_col: str, cl: int, time_col: str, event_col: str,
                                    covariate_cols=None, stem_suffix=""):
    """Calculates RMST stats in R and plots KM curves for an individual cluster."""
    covariate_cols = COVARIATE_COLS if covariate_cols is None else covariate_cols
    tmp = df_cluster.copy()
    tmp["is_cluster"] = (tmp[cluster_col] == cl).astype(int)

    use_cols = [time_col, event_col, "is_cluster"] + covariate_cols
    tmp = tmp[use_cols].dropna()

    n_rest, n_cluster = tmp['is_cluster'].value_counts().sort_index()

    # Prepping columns to be legally valid names for R
    r_cov_names =[c.replace(" ", "_").replace("(", "").replace(")", "") for c in covariate_cols]
    r_df = tmp.copy()
    r_df.columns = ["time", "event", "arm"] + r_cov_names

    adjust = len(covariate_cols) > 0

    try:
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv["surv_data"] = ro.conversion.py2rpy(r_df)
        ro.globalenv["tau"] = TAU_YEARS

        ro.r("library(survRM2)")
        ro.r("rmst_unadj <- rmst2(time=surv_data$time, status=surv_data$event, arm=surv_data$arm, tau=tau)")

        unadj = np.array(ro.r("rmst_unadj$unadjusted.result[1,]"))
        rmst_cluster = float(ro.r("rmst_unadj$RMST.arm1$rmst['Est.']")[0])
        rmst_rest = float(ro.r("rmst_unadj$RMST.arm0$rmst['Est.']")[0])
        diff_unadj, ci_low_unadj, ci_high_unadj, p_unadj = unadj

        print(f"\n{'='*60}\nUNADJUSTED RMST (Cluster {cl} vs Rest)\n{'='*60}")
        print(f"  RMST Cluster {cl}: {rmst_cluster:.3f} yr ({rmst_cluster*12:.1f} months)")
        print(f"  RMST Rest:       {rmst_rest:.3f} yr ({rmst_rest*12:.1f} months)")
        print(f"  Difference:      {diff_unadj:+.3f} yr ({diff_unadj*12:+.1f} months)")
        print(f"  p-value:         {p_unadj:.4f}")

        if adjust:
            cov_r_str = ", ".join([f'"{c}"' for c in r_cov_names])
            ro.r(f"""
            cov_mat <- as.matrix(surv_data[, c({cov_r_str})])
            rmst_adj <- rmst2(time=surv_data$time, status=surv_data$event, arm=surv_data$arm, tau=tau, covariates=cov_mat)
            """)
            adj = np.array(ro.r("rmst_adj$adjusted.result[1,]"))
            diff_adj, ci_low_adj, ci_high_adj, p_adj = adj


            cov_label = " + ".join(covariate_cols)

            print(f"\nADJUSTED RMST ({cov_label}, n={len(tmp)})")
            print(f"  Adj difference: {diff_adj:+.3f} yr ({diff_adj*12:+.1f} months)")
            print(f"  Adj p-value:    {p_adj:.4f}")
        else:
            diff_adj, ci_low_adj, ci_high_adj, p_adj, cov_label = diff_unadj, ci_low_unadj, ci_high_unadj, p_unadj, "none"

    except Exception as e:
        print(f"RMST failed for Cluster {cl}: {e}")
        return None

    # --- Plot Kaplan-Meier via Python's lifelines ---
    fig, ax = plt.subplots(figsize=(8, 6))
    km_colors = {1: COLOR_CLUSTER, 0: COLOR_REST}
    km_labels = {1: f"Cluster {cl}", 0: 'Rest'}
    km_tables = []

    for grp in [1, 0]:
        mask = tmp["is_cluster"] == grp
        kmf = KaplanMeierFitter()
        kmf.fit(tmp.loc[mask, time_col], tmp.loc[mask, event_col], label=km_labels[grp])
        kmf.plot_survival_function(ax=ax, color=km_colors[grp], linewidth=2.5, ci_show=True, ci_alpha=0.15)
        km_tables.append(pd.concat([kmf.survival_function_, kmf.confidence_interval_], axis=1)
                         .set_axis(["survival", "ci_lower", "ci_upper"], axis=1).assign(group=km_labels[grp]))

        censored_times = tmp.loc[mask & (tmp[event_col] == 0), time_col]
        for t in censored_times:
            ax.plot(t, kmf.predict(t), "|", color=km_colors[grp], markersize=8, markeredgewidth=1.5)
    rasterize_ci_bands(ax)

    ax.axvline(TAU_YEARS, color="gray", linestyle="--", linewidth=1.2, alpha=0.7, label=f"τ={TAU_YEARS}yr")

    annot = (
        f"RMST (τ={TAU_YEARS}yr):\n"
        f"  Cluster {cl}: {rmst_cluster:.2f} yrs\n"
        f"  Rest: {rmst_rest:.2f} yrs\n"
        f"  Δ: {diff_adj*12:+.1f} months\n"
        f"  p(adj) = {p_adj:.4f}"
    )

    ax.text(0.97, 0.97, annot, transform=ax.transAxes, fontsize=12, va="top", ha="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))

    ax.set_title(f"{ENDPOINT} Survival: Cluster {cl} (n={n_cluster}) vs Rest (n={n_rest})", fontsize=14, fontweight="bold")
    ax.set_xlabel("Time (years)", fontsize=12)
    ax.set_ylabel("Survival probability", fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0, None)
    ax.legend(loc="lower left", frameon=False, fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
    
    plt.tight_layout()

    # Save output; cluster 2 vs rest is Fig 3c
    panel = "Fig 3c" if (cl == 2 and covariate_cols == COVARIATE_COLS) else None
    output.save(OUT, f"Cluster_{cl}_vs_Rest_{ENDPOINT}_RMST_KM{stem_suffix}", fig,
                pd.concat(km_tables).rename_axis("time_years"), panel=panel)
    plt.show()
    plt.close(fig)

    return {
        "cluster": cl, "n_cluster": n_cluster, "n_rest": n_rest, "tau_yr": TAU_YEARS,
        "rmst_cluster_yr": rmst_cluster, "rmst_rest_yr": rmst_rest,
        "diff_months_unadj": diff_unadj * 12, "ci_months_low_unadj": ci_low_unadj * 12, "ci_months_high_unadj": ci_high_unadj * 12, "p_unadj": p_unadj,
        "diff_months_adj": diff_adj * 12, "ci_months_low_adj": ci_low_adj * 12, "ci_months_high_adj": ci_high_adj * 12, "p_adj": p_adj,
    }



def plot_global_km_curves(df_cluster: pd.DataFrame, cluster_col: str, time_col: str, event_col: str):
    """Outputs Figure 3c KM logic plotting multivariate lines simultaneously"""
    df_km = df_cluster[[cluster_col, time_col, event_col]].copy()
    df_km[cluster_col] = df_km[cluster_col].astype(str)
    # kmplot expects signed durations in days (negative = censored)
    df_km["OS.signed.days"] = np.where(df_km[event_col] == 1, 1, -1) * df_km[time_col] * AVG_NUM_DAYS_PER_YEAR
    hue_order = sorted(df_km[cluster_col].unique(), key=lambda x: int(x))

    # Single KM
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    kmplot(data=df_km, x="OS.signed.days", hue=cluster_col, hue_order=hue_order,
           convert_days_to_months=True, add_hazard_ratio=False, add_logrank_pvalue=False,
           at_risk_counts=False, ax=ax, title="Overall survival by WES CNV Leiden cluster")
    rasterize_ci_bands(ax)
    plt.tight_layout()
    output.save(OUT, "km_all_clusters", fig)
    plt.show()

    # KM + Log Rank Text
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    kmplot(data=df_km, x="OS.signed.days", hue=cluster_col, hue_order=hue_order,
           convert_days_to_months=True, add_hazard_ratio=False, add_logrank_pvalue=False,
           at_risk_counts=False, ax=ax, title="Overall survival by WES CNV Leiden cluster")
    rasterize_ci_bands(ax)

    df_lr = df_km.dropna(subset=[time_col, cluster_col]).copy()
    res = multivariate_logrank_test(event_durations=df_lr[time_col], groups=df_lr[cluster_col], event_observed=df_lr[event_col])

    p = float(res.p_value)
    p_txt = "Global log-rank: p<0.001" if p < 0.001 else f"Global log-rank: p={p:.3g}"
    ax.text(0.02, 0.02, p_txt, transform=ax.transAxes, fontsize=12, va="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))
    
    plt.tight_layout()
    output.save(OUT, "km_all_clusters_logrank", fig)
    plt.show()
    return p


def row(label, r, n_rows):
    """Cluster 2 vs rest line of an RMST summary table."""
    r = r.loc[r["cluster"].astype(int) == 2].iloc[0]
    return dict(model=label, n_joined_rows=n_rows, n_model=int(r["n_cluster"] + r["n_rest"]),
                n_cluster2=int(r["n_cluster"]), n_rest=int(r["n_rest"]),
                diff_months=r["diff_months_adj"], ci_low=r["ci_months_low_adj"], ci_high=r["ci_months_high_adj"],
                p=r["p_adj"], diff_months_unadj=r["diff_months_unadj"], p_unadj=r["p_unadj"])

In [ ]:
# ===== USER CONFIG =====
OUT = output.setup("fig3c_suppfig3b_wes_profile_os_rmst")
TAU_YEARS = 3
ENDPOINT = "OS"
CLINICAL_ID_COL = "Subject Id"
COVARIATE_COLS = ["Age (years)", "MGMT promoter methylation", "Administrative gender", "p53"]
# The model of the published Fig 3c (site coded as one ordinal covariate CH 0, ER 1, CR 2)
PUBLISHED_COVARIATE_COLS = ["Age (years)", "MGMT promoter methylation", "Administrative gender", "Site code"]
PUBLISHED = dict(n_rows=253, n_model=247, diff=3.3532, lo=0.2770, hi=6.4293, p=0.0326)  # cluster 2 vs rest

# Plot aesthetics
figsize = (9, 6)
title_fontsize = 8
axis_label_fontsize = 12
tick_fontsize = 12
legend_fontsize = 20
annot_fontsize = 12
km_linewidth = 2
censor_markersize = 6
censor_markeredgewidth = 1.5

CLUSTER_CSV = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/WES/infercnv/leiden_cluster_wes_baseline_samples.csv"

COLOR_CLUSTER = "#fe7a02"    # orange
COLOR_REST = "#007ebc"     # blue


# Endpoint mapping
if ENDPOINT.upper() == "OS":
    time_col, event_col = "OS.time", "OS.event"
    time_label = "Overall Survival"
elif ENDPOINT.upper() == "PFS":
    time_col, event_col = "PFS.time", "PFS.event"
    time_label = "Progression-Free Survival"
else:
    raise ValueError("ENDPOINT must be 'OS' or 'PFS'")

In [ ]:
df_clinical = load_clinical_data()
print(f"Clinical rows: {len(df_clinical)}")
df_cluster, cluster_col = load_and_merge_cluster_data(df_clinical,endpoint=ENDPOINT)

clusters = sorted(df_cluster[cluster_col].unique())
print(f"Joined rows: {len(df_cluster)} ({patient_of(df_cluster.index).nunique()} patients); "
      f"missing values: {df_cluster[COVARIATE_COLS].isna().sum().to_dict()}")

print("\n" + "#" * 60)
print("1. Survival Analysis (Cluster vs Rest)")
print("#" * 60)

# 1B: Loop over RMST and generate Cluster vs Rest individual Plots 
rmst_results = []
for cl in clusters:
    print(f"\n{'#'*60}\nRunning RMST for Cluster {cl} vs Rest\n{'#'*60}")
    res = run_rmst_and_km_cluster_vs_rest(df_cluster, cluster_col, cl, time_col=time_col, event_col=event_col)
    if res:
        rmst_results.append(res)

# Save cumulative RMST summary
if rmst_results:
    df_rmst_summary = pd.DataFrame(rmst_results)
    output.save_table(OUT, "cluster_vs_rest_rmst_summary", df_rmst_summary, index=False)
    print(df_rmst_summary.to_string(index=False))

print("\n" + "#" * 60)
print("3. Global Survival Plots (Figure 3c)")
print("#" * 60)

# 3A: Global Kaplan-Meier and Log-rank test
p_global = plot_global_km_curves(df_cluster, cluster_col, time_col=time_col, event_col=event_col)
print(f"Global log-rank p = {p_global:.5f} (n = {len(df_cluster)})")

print("\nPipeline execution complete.")

# Forest plot paper fig (Supp Fig 3b)

In [ ]:
df_rmst_summary['cluster'] = df_rmst_summary['cluster'].astype(str)
with mock.patch.object(plt, "show"):  # forest_plot ends with plt.show(), which closes the figure in a notebook
    forest_plot(df_rmst_summary)
output.save(OUT, "fig3e_rmst_forest_plot", plt.gcf(), df_rmst_summary, panel="Supp Fig 3b")
plt.show()

# Published model (age, MGMT, sex, site)

The same join and code with the covariates of the published Fig 3c; must give cluster 2 vs
rest +3.35 months [0.28, 6.43], p = 0.0326, n = 247 of 253 joined rows.

In [ ]:
df_cluster_pub, _ = load_and_merge_cluster_data(df_clinical, endpoint=ENDPOINT,
                                                covariate_cols=PUBLISHED_COVARIATE_COLS)
published_results = []
for cl in clusters:
    res = run_rmst_and_km_cluster_vs_rest(df_cluster_pub, cluster_col, cl, time_col=time_col, event_col=event_col,
                                          covariate_cols=PUBLISHED_COVARIATE_COLS, stem_suffix="_published_model")
    published_results.append(res)
df_rmst_published = pd.DataFrame(published_results)
output.save_table(OUT, "cluster_vs_rest_rmst_summary_published_model", df_rmst_published, index=False)
print(df_rmst_published.to_string(index=False))

# Model results

Cluster 2 vs rest, τ = 3 years: model n, ΔRMST (months), 95% CI and p of the model above and
of the published model, next to the published numbers.

In [ ]:
table = pd.DataFrame([
    row("age + MGMT + sex + p53", df_rmst_summary, len(df_cluster)),
    row("published: age + MGMT + sex + site (ordinal)", df_rmst_published, len(df_cluster_pub)),
    dict(model="published numbers", n_joined_rows=PUBLISHED["n_rows"], n_model=PUBLISHED["n_model"],
         diff_months=PUBLISHED["diff"], ci_low=PUBLISHED["lo"], ci_high=PUBLISHED["hi"], p=PUBLISHED["p"]),
])
output.save_table(OUT, "fig3c_cluster2_models", table.round(4), index=False)
print(table.round(4).to_string(index=False))

# Versions

In [ ]:
print(output.versions(["numpy", "pandas", "matplotlib", "lifelines", "rpy2"], ["survRM2"]).to_string(index=False))